# When Imbalance Correction Backfires
### Robustness of Tabular Imbalance Mitigation Under Minority-Label Contamination

**Source document:** `finaldraft.pdf` (Research Project Blueprint, prepared Aug 12, 2026)

**Status of source doc:** Experimental protocol only — no empirical results have been generated yet. This notebook is a working reference for the plan, plus placeholder cells to build the implementation into as you go.

---

This notebook summarizes the blueprint step by step, then provides empty skeleton cells matching the paper's proposed repository structure (Section 13.1) so code can be added directly here as the study progresses.

## 1. The Core Question

Standard imbalance-correction techniques (SMOTE, oversampling, class weighting, TabPFN threshold correction, DistPFN) all implicitly assume that when a training example is labeled "minority class," that label is trustworthy.

**This study asks:** what happens to these corrections when that assumption breaks — i.e., when the rare-class labels themselves are corrupted, not just imbalanced? Specifically: *when do imbalance corrections help, stop helping, or become actively harmful, and does the answer differ depending on the direction of the label corruption?*

### Why it's novel (without overclaiming)
The paper explicitly does **not** claim that "class imbalance + label noise" is a new problem — CLIMB, TILBench, and Robust-GBDT already cover that broadly. The narrower, defensible contribution is testing whether the **correction mechanism itself** becomes unreliable under two specific, directed rare-label error processes, and identifying the point at which each correction crosses from helpful to harmful.

## 2. The Two Directed Label-Error Mechanisms

The central methodological idea: split label corruption into two mechanisms and analyze them **separately**, since they are expected to break corrections in different ways.

| | Direction A — Majority → Minority Contamination | Direction B — Minority → Majority Label Loss |
|---|---|---|
| **What happens** | Majority examples get mislabeled as minority | True minority examples get mislabeled as majority |
| **Effect on observed prior** | Inflated (looks like more minority examples exist) | Deflated (looks like fewer minority examples exist) |
| **Risk to corrections** | SMOTE/oversampling/weighting may amplify the false minority examples | Genuine rare examples are lost before correction can use them |
| **c parameter meaning** | Fraction of the *observed* minority-labeled pool that is actually majority | Fraction of *true* minority labels that are omitted |

`c = 0` is a single shared clean condition used by both directions.

## 3. Controlled Experimental Grid

| Factor | Levels |
|---|---|
| Datasets | 5 public UCI datasets (Banknote Authentication, Blood Transfusion, South German Credit, Spambase, Breast Cancer Wisconsin Diagnostic) |
| True minority prevalence *r* | 0.50, 0.20, 0.10, 0.05 |
| Directed noise level *c* | 0, 0.05, 0.10, 0.20, 0.30 (per direction; c=0 shared) |
| Label-error direction | A (maj→min), B (min→maj) |
| Random seeds | 13, 37, 73 |
| Correction pipelines | 9 core + oracle-prior diagnostics |

**Key design controls:**
- Constant clean training size (`N_common`) across all imbalance levels *r*, so imbalance is never confounded with total sample size.
- A single clean, untouched test set per dataset/seed, shared across every condition.
- Corrections are applied *after* the directed label error, exactly as a practitioner would encounter it.

**Estimated workload:** 540 unique dataset-seed-stress contexts → ~3,240 fit/inference jobs → ~4,860 evaluated method-condition rows.

## 4. Correction Strategies Under Test

| Pipeline | What it does |
|---|---|
| TabPFN – none | Uncorrected TFM reference |
| TabPFN – threshold | Classify minority if p(y=1) > observed minority prior (McDowell et al.) |
| TabPFN – downsample | Downsample majority to 1:1 before running TabPFN |
| DistPFN | Posterior adjustment using observed training prior |
| DistPFN-T | Adaptive/temperature-scaled version of DistPFN |
| XGBoost – none | Classical baseline, no correction |
| XGBoost – weighted | `scale_pos_weight` from observed noisy class counts |
| XGBoost – random oversampling | Duplicate observed minority rows |
| XGBoost – SMOTE | Synthetic minority oversampling on training fold only |

**Oracle-prior diagnostic (not deployable):** for threshold/DistPFN/DistPFN-T, rerun using the *true* pre-contamination prior instead of the corrupted observed prior. This isolates how much of any correction failure is caused specifically by prior distortion.

## 5. New Contribution Metrics (Section 10 of the blueprint)

These are the paper's real empirical contribution beyond just running experiments:

- **Correction Gain (CG)** — `score(corrected) - score(uncorrected)` for the same dataset/seed/r/c/direction cell. Sign-flipped for loss metrics so positive always means "helped."
- **Correction Harm Rate (CHR)** — fraction of evaluated cells where CG < 0 (i.e., how *often* a correction hurts).
- **Correction Break-Even Point (c\*)** — smallest noise level *c* at which CG drops to ≤ 0 for a given method/direction.
- **Regret** — `max(0, uncorrected_score - corrected_score)` (how *badly* it hurts when it does).
- **Prior Sensitivity Gap (PSG)** — `oracle_prior_score - observed_prior_score`; a positive gap implicates prior corruption as the cause of failure.

## 6. Evaluation Metrics

**Discrimination (primary):** Balanced Accuracy, Worst-Class Accuracy/Recall, AUPRC, Macro-F1

**Diagnostic:** Minority precision/recall, AUROC (secondary — can conceal rare-class failure)

**Reliability:** Brier score, Log loss, Expected Calibration Error (ECE)

Hypothesis H5: calibration/reliability metrics often reveal correction harm *before* AUROC does — i.e., a model can keep ranking well while its probabilities become untrustworthy.

## 7. Three-Day Execution Schedule

| Day | Objective | Stop condition |
|---|---|---|
| **Day 1** | Freeze protocol; get one dataset running end-to-end | `raw_runs.csv` produced correctly for one dataset × one seed × all 36 stress cells |
| **Day 2** | Complete evidence and freeze results | Full core matrix run, both directions; CG/CHR/regret/break-even/PSG computed; all figures/tables generated |
| **Day 3** | Write the manuscript around verified results | Methods → Results → Intro/Related Work → Discussion → Threats → Abstract (last); every number traced to CSV |

**Runtime fallback rule:** if compute becomes the bottleneck, drop optional methods (CatBoost/ITBoost) first, then oracle diagnostics — never cherry-pick datasets based on performance.

## 8. Reproducibility & Research-Integrity Rules

- No result value enters the paper until it exists in the frozen results files — no placeholder/hypothetical numbers.
- No test-set tuning; the clean test set is only for final evaluation.
- No post-hoc model/checkpoint switching after results are seen.
- No silent failure deletion — `failures.csv` is part of the reproducibility package.
- No cherry-picking — dataset/method removal must be pre-defined and documented *before* comparing results.
- Every noisy run must log its label-error direction, requested/realized rate, flipped-label indices, and prior values.
- Literature search must be refreshed immediately before submission.

## 9. Datasets

| Dataset | UCI ID | N | Features | Domain | Link | DOI |
|---|---|---|---|---|---|---|
| Banknote Authentication | 267 | 1,372 | 4 | Counterfeit vs genuine banknote | [archive.ics.uci.edu/dataset/267](https://archive.ics.uci.edu/dataset/267/banknote+authentication) | 10.24432/C55P57 |
| Blood Transfusion Service Center | 176 | 748 | 4 | Donor donated in target period | [archive.ics.uci.edu/dataset/176](https://archive.ics.uci.edu/dataset/176/blood+transfusion+service+center) | 10.24432/C5GS39 |
| South German Credit | 522 | 1,000 | 20 | Credit risk: good vs bad | [archive.ics.uci.edu/dataset/522](https://archive.ics.uci.edu/dataset/522/south+german+credit) | 10.24432/C5X89F |
| Spambase | 94 | 4,601 | 57 | Spam vs non-spam email | [archive.ics.uci.edu/dataset/94](https://archive.ics.uci.edu/dataset/94/spambase) | 10.24432/C53G6X |
| Breast Cancer Wisconsin (Diagnostic) | 17 | 569 | 30 | Malignant vs benign diagnosis | [archive.ics.uci.edu/dataset/17](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic) | 10.24432/C5DW2B |

**Note:** South German Credit (UCI 522) is not importable via the `ucimlrepo` Python package (`fetch_ucirepo(id=522)` raises `DatasetNotFoundError`). Its actual data file must be downloaded directly from `https://archive.ics.uci.edu/static/public/522/south+german+credit.zip` and parsed from the bundled `SouthGermanCredit.asc` file instead.

## 10. Key References

- **CLIMB** — Liu et al., NeurIPS 2025 (Datasets & Benchmarks), arXiv:2505.17451
- **TILBench** — Liu & Luo, arXiv:2605.14915, May 2026
- **PFN imbalance correction (threshold/downsample)** — McDowell et al., arXiv:2605.21742v1, May 2026
- **DistPFN / DistPFN-T** — Lee et al., ICML 2026; arXiv:2605.04363v2
- **Noise Immunity in TabPFN** — Hu & Ghelichi, arXiv:2604.04868, Apr 2026
- **ITBoost** — Su et al., arXiv:2605.04671, May 2026
- **Uncertainty benchmark** — De Melo Costa et al., ESANN 2026
- **SkillTFM** — He et al., arXiv:2608.06137, Aug 2026

Full reference list with DOIs is in Section 19 of `finaldraft.pdf`.

## 11. Next Steps — Step-by-Step Implementation Plan

Adapted from the blueprint's execution protocol (Section 14.1, Steps 1–45), reorganized into 12 phases and mapped onto the notebook sections in **Implementation Skeleton** below (10.1–10.10) so each phase has a home to write code in. Everything happens inside this notebook — no separate repo/`src/` files. Check items off as they're done; log what actually happened in the **Daily Diary** section at the bottom.

### Phase 1 — Environment (→ notebook section 10.1)
- [ ] Install in this kernel: numpy, pandas, scipy, scikit-learn, matplotlib, xgboost, tabpfn, imbalanced-learn, ucimlrepo, statsmodels, tqdm, pyyaml
- [ ] Record environment identity in a cell: package versions, OS, CPU/GPU, TabPFN checkpoint/model identifier, and the three seeds (13, 37, 73) *(Step 4)*
- [ ] Freeze this identity before looking at any comparative results — never change the TabPFN checkpoint after seeing results

### Phase 2 — Data (→ 10.2)
- [ ] Fetch UCI IDs 267, 176, 522, 94, 17 (522 needs the direct-zip workaround documented in Section 9 above) *(Step 5)*
- [ ] Audit each dataset: row count, feature count, dtypes, duplicate-row count, missing values per column, raw target values/counts *(Step 6)*
- [ ] Resolve target semantics from official docs — not assumption — and fix one positive/minority class per dataset, encoded as 1 *(Step 7)*
- [ ] Save a machine-readable dataset manifest (raw counts, missing values, any transformations) *(Section 4.1)*

### Phase 3 — Splits & Controlled Imbalance (→ 10.3)
- [ ] For seeds 13/37/73, create one stratified 80/20 clean train/test split per dataset, **before** any imbalance or label error *(Step 8)*
- [ ] Freeze the test set: never resample, corrupt, or tune on it; must stay byte-identical across every r/c/direction/method *(Step 9)*
- [ ] Compute `N_common` per dataset/seed so every r uses the same total clean training size *(Step 10)*
- [ ] Build each controlled imbalance set for r ∈ {0.50, 0.20, 0.10, 0.05} by sampling `N_common` rows *(Step 11)*
- [ ] Verify: exact size, correct minority/majority counts, no overlap with test set, identical `N_common` across r *(Step 12)*

### Phase 4 — Directed Label-Error Injection (→ 10.4)
- [ ] For c=0, store one shared clean condition (no direction-specific duplicate needed)
- [ ] For c ∈ {0.05, 0.10, 0.20, 0.30}, generate **both** Direction A and Direction B noisy-label copies from the same clean controlled rows *(Step 13)*
- [ ] Verify mathematically: at c=0 labels are identical; Direction A → π_obs ≥ π_true; Direction B → π_obs ≤ π_true *(Step 14)*
- [ ] Confirm every competing correction method reuses the identical noisy labels for a given dataset/seed/r/c/direction cell — never redraw per method *(Step 15)*

### Phase 5 — Leakage-Safe Preprocessing
- [ ] Fit numeric median imputation / categorical imputation+one-hot encoding on the current training condition only; apply to the fixed test set; unknown test categories are ignored, never learned *(Step 16)*

### Phase 6 — Correction Methods: TabPFN Family (→ 10.5)
- [ ] Implement frozen TabPFN baseline; save raw P(y=1) on test rows so post-hoc corrections can reuse it *(Step 17)*
- [ ] Implement threshold correction (classify minority if p > π_obs) + oracle-π_true diagnostic, clearly labeled non-deployable *(Step 18)*
- [ ] Implement TabPFN downsampling (balance observed majority to 1:1, rerun TabPFN) *(Step 19)*
- [ ] Implement DistPFN exactly per the paper formula; unit-test probabilities sum to 1; sanity-check against official code on a toy example *(Step 20)*
- [ ] Implement DistPFN-T exactly per the paper formula; **validate against official code before full runs** *(Step 21)*
- [ ] Wire up observed-prior vs oracle-prior diagnostic variants for threshold/DistPFN/DistPFN-T → this becomes the Prior Sensitivity Gap *(Step 22)*

### Phase 7 — Correction Methods: XGBoost Family (→ 10.6)
- [ ] Implement frozen XGBoost baseline — one fixed config used for every variant, no per-condition tuning *(Step 23)*
- [ ] Implement weighted XGBoost (`scale_pos_weight` from observed noisy class counts) *(Step 24)*
- [ ] Implement RandomOverSampler + XGBoost (training-fold only) *(Step 25)*
- [ ] Implement SMOTE + XGBoost (training-fold only; deterministic `k_neighbors` reduction or mark the cell failed if mathematically infeasible, never silently change the design) *(Step 26)*
- [ ] Optional, only after the core matrix exists: CatBoost/ITBoost sanity baselines *(Step 27)*

### Phase 8 — Metrics Module (→ 10.7)
- [ ] Implement balanced accuracy, worst-class accuracy/recall, AUPRC, macro-F1, minority precision/recall, AUROC, Brier score, log loss, ECE (fixed bin count, documented) in one tested module *(Step 28)*
- [ ] Define score direction consistently: for loss-type metrics (Brier/log loss), flip the sign so positive gain always means "correction helped" *(Step 29)*

### Phase 9 — Contribution Metrics (→ 10.8)
- [ ] Compute Correction Gain, Correction Harm Rate, Regret, break-even c*, and Prior Sensitivity Gap — separately per direction, before any cross-direction synthesis *(Step 37, Section 10)*

### Phase 10 — Unit Tests & Smoke Test (→ 10.9)
- [ ] Run all unit tests: c=0 identity, realized-rate tolerance, `N_common` consistency across r, no train/test overlap, DistPFN normalization *(Step 30)*
- [ ] One-cell smoke test: one dataset, one seed, one r, one nonzero c, both directions, all core methods + the shared c=0 cell *(Step 31)*
- [ ] One-dataset pilot: all 36 unique stress cells × 3 seeds for one dataset — inspect only for bugs/leakage/runtime, **not** to pick a favorite method *(Step 32)*
- [ ] Freeze the protocol (config + notebook state/commit) once the pilot is technically correct — any later change must be documented with a reason *(Step 33)*

### Phase 11 — Full Run & Analysis (→ 10.10)
- [ ] Run the full core matrix: 5 datasets × 3 seeds × 36 stress contexts × 9 pipelines (540 stress contexts, ~4,860 evaluated method-context rows) *(Step 34)*
- [ ] Log failures to a `failures` table with method/dataset/condition/reason rather than silently dropping cells *(Step 35)*
- [ ] Build the frozen summary table — aggregate seeds within each dataset-method-r-c-direction cell *(Step 36)*
- [ ] Run planned statistics: paired-block Friedman omnibus test, then paired Wilcoxon signed-rank tests with Holm correction; report effect sizes, not just p-values *(Step 38)*
- [ ] Generate all required figures/tables (Section 12 of the blueprint) directly from the frozen data — never hand-typed values *(Step 39)*
- [ ] Interpret without forcing the hypothesis — report negative or dataset-heterogeneous results exactly as they come out *(Step 40)*

### Phase 12 — Manuscript (Day 3 equivalent)
- [ ] Write Methods first — must be reproducible without looking at the result tables *(Step 41)*
- [ ] Write Results directly from frozen outputs, answering RQ1–RQ5 in order, keeping Direction A/B separate unless an aggregate is explicitly defined *(Step 42)*
- [ ] Write Discussion with bounded claims — distinguish "correction hurts" from "base model fails," compare to McDowell/DistPFN/noise literature *(Step 43)*
- [ ] Refresh the literature search immediately before submission; revise the novelty paragraph if something new overlaps *(Step 44)*
- [ ] Final reproducibility/integrity audit — rerun sampled cells, rebuild figures from raw results, verify citations, remove any placeholder values *(Step 45)*

---
## Implementation Skeleton

The cells below mirror the proposed repository structure (Section 13.1). Fill these in as you start Day 1 work. They are currently empty placeholders.

### 10.1 Environment setup

In [ ]:
# TODO: imports (numpy, pandas, scipy, scikit-learn, xgboost, tabpfn, imbalanced-learn, ucimlrepo, statsmodels)
# TODO: record pip freeze, OS, CPU/GPU, TabPFN checkpoint identifier, random seeds


### 10.2 Dataset loading and audit (Section 4)

In [ ]:
# TODO: load the 5 UCI datasets (267, 176, 522, 94, 17)
# TODO: verify class counts, missing values, identifier columns; build dataset manifest


### 10.3 Constant training size + controlled imbalance (Section 5.2)

In [ ]:
# TODO: compute_common_training_size(r_values, n_min_available, n_maj_available)
# TODO: sample N_common rows at each target r


### 10.4 Directed label-error functions (Section 6 / 13.4)

In [ ]:
# TODO: def majority_to_minority_contamination(y_clean, c, rng): ...
# TODO: def minority_to_majority_loss(y_clean, c, rng): ...


### 10.5 Correction methods — TabPFN family (Section 13.5)

In [ ]:
# TODO: tabpfn_none, tabpfn_threshold (observed + oracle), tabpfn_downsample
# TODO: DistPFN and DistPFN-T (validate against official implementation on a toy example first)


### 10.6 Correction methods — XGBoost family (Section 13.6)

In [ ]:
# TODO: xgb_none, xgb_weighted (scale_pos_weight), xgb_ros (RandomOverSampler), xgb_smote


### 10.7 Metrics module (Section 9)

In [ ]:
# TODO: balanced_accuracy, worst_class_accuracy/recall, auprc, macro_f1,
#       minority precision/recall, auroc, brier, log_loss, ece


### 10.8 Contribution metrics — gain / harm / break-even / PSG (Section 10)

In [ ]:
# TODO: correction_gain, correction_harm_rate, break_even_point, regret, prior_sensitivity_gap


### 10.9 Unit tests before full run (Section 13.8)

In [ ]:
# TODO: c=0 returns exactly clean labels for both directions
# TODO: realized noise rate within rounding tolerance of requested c
# TODO: N_common identical across r; test rows/labels never change across conditions
# TODO: DistPFN implementation matches paper formula + official-code sanity check


### 10.10 Run matrix + analysis (Day 1–2)

In [ ]:
# TODO: smoke test -> one-dataset pilot -> freeze protocol hash -> full core matrix
# TODO: write raw_runs.csv, failures.csv, then build summary.csv deterministically


---
## Daily Diary

This section is a running log of actual work done, updated as we go. Newest entries at the bottom.

### 2026-08-16 — Day 1 Summary

This notebook stayed the planning/reference document; the actual working code now lives in **`backup.ipynb`** (a copy of this file that got upgraded cell-by-cell). Logging what happened across the session here so this stays a complete record:

**Dataset links added** to Section 9 above — verified via the UCI API rather than guessed, including the South German Credit workaround (not importable via `ucimlrepo`; fetched directly from UCI's static zip).

**Section 11 ("Next Steps") added** — a 12-phase checklist adapted from the blueprint's Section 14.1 execution protocol (Steps 1–45), mapped onto the Implementation Skeleton's 10.1–10.10 sections below.

**`data/` folder created** at the project root with everything needed to start modeling:
- `data/raw/{slug}/` — untouched original files per dataset (Banknote, Blood Transfusion, South German Credit, Spambase, Breast Cancer Wisconsin), named descriptively rather than by numeric UCI ID
- `data/processed/{slug}_{X,y,full}.csv` — clean feature matrix + encoded target (minority=1) per dataset
- `data/manifest.json` — machine-readable audit: counts, missing values, duplicates, and the cited source for each positive-class decision (including the flagged Banknote caveat — UCI's own metadata doesn't document its 0/1 mapping)

**`backup.ipynb` upgraded to working code** — every `# TODO` cell in that copy's Implementation Skeleton (10.1–10.10) was replaced with real, executed code reading from `data/`: environment setup, dataset audit, `N_common`/controlled-imbalance sampling, leakage-safe preprocessing (added as 10.3b — wasn't in the original skeleton), both directed label-error functions, TabPFN correction math (model calls deferred — `tabpfn` isn't installed in the kernel yet), all 4 XGBoost pipelines, the metrics module, contribution metrics, 6 passing unit tests, and a one-cell smoke test on real data (Banknote, seed 13, r=0.20, c=0.20) — all verified by actually executing the notebook end-to-end via `jupyter nbconvert --execute`.

**This notebook's own Implementation Skeleton (below) is intentionally left as TODO placeholders** — it stays the clean reference copy. See `backup.ipynb`'s diary for the full account of a cell-positioning bug hit and fixed during that upgrade (an inserted cell pair shifted subsequent cells by one slot because the original cells had no stable `id` fields; fixed by rewriting the raw JSON and assigning every cell a permanent id).

**Next up:** install `tabpfn` in the kernel, then the one-dataset pilot (all 36 stress cells × 3 seeds) in `backup.ipynb`.